# A07 · 上下文定向、扩量与数据边界

先修：公用31–34；A06。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

广告主选择目标上下文，平台在有限字段中匹配。DMP是一种管理分群与数据使用的角色，不意味着可以任意跨域拼接身份。这里仅保留page_context字段，不存设备标识或敏感属性。冷启动可以使用当前页面上下文；扩量扩大覆盖，也可能降低精确度。人工相关集合只是机制标签。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
strict=ac.targeting();expanded=ac.targeting(True)
display(pd.DataFrame([strict,expanded],index=['strict','expanded']))
assert expanded['coverage']>strict['coverage']
assert expanded['precision']<strict['precision']
assert strict['fields']==['page_context']

,selected,coverage,precision,fields
strict,"[0, 3]",0.4,1.000000,[page_context]
expanded,"[0, 1, 3]",0.6,0.666667,[page_context]


## 解释与局限

一次扩量只改变允许的上下文集合。更大覆盖不自动意味着更多真实转化，人工相关标签也不能验证人群模型。Criteo的匿名类别没有已知语义，不在这里冒充年龄、兴趣或跨设备身份。

## 查看源码，再做练习

增加未知上下文并设计冷启动回退；先写允许字段与保留期限，再决定是否需要额外数据。

In [3]:
import inspect
print(inspect.getsource(ac.targeting))

def targeting(expand=False):
    # Context is deliberately provided; no device ID, inferred sensitive label or join.
    contexts=['运动','生活','旅行','运动','科技']
    allowed={'运动','生活'} if expand else {'运动'}
    chosen=[i for i,c in enumerate(contexts) if c in allowed]
    relevant={0,3}
    return dict(selected=chosen,coverage=len(chosen)/len(contexts),precision=len(set(chosen)&relevant)/max(1,len(chosen)),fields=['page_context'])



## 来源与下一步

A-RTB 133–148页；字段最小化是Lab展开。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。